# Unit Tests for GW Chat3 Plot Functions

This notebook contains unit tests for:
- `plot_indicator` function
- Dropbox upload functionality
- Image display functions
-# Copy functions from GW_chat3 for testing
# In production, these would be in separate .py files


In [ ]:
# Import required libraries for testing
import unittest
from unittest.mock import Mock, patch, MagicMock
import os
import sys
from io import BytesIO
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

# Add the current directory to path to import functions
sys.path.append('.')

# Import functions from the main notebook
# Note: In a real scenario, you'd import from a .py file
# For notebook testing, we'll need to copy the functions or use exec


In [ ]:
# Copy functions from main notebook for testing
# In production, these would be in separate .py files

import dropbox
from dotenv import load_dotenv
import base64

load_dotenv(override=True)

def upload_image_to_dropbox(image_data, filename):
    """Upload image to Dropbox and return shareable link"""
    try:
        # Get access token from environment variables
        access_token = os.getenv('DROPBOX_ACCESS_TOKEN')
        if not access_token:
            return "Error: DROPBOX_ACCESS_TOKEN not found in environment variables"
        
        # Initialize Dropbox client
        dbx = dropbox.Dropbox(access_token)
        
        # Upload file to Dropbox app folder
        # Automatically uploads to /Apps/YourAppName/filename
        dbx.files_upload(image_data, f'/{filename}')
        
        # Create shareable link (correct method name)
        shared_link = dbx.sharing_create_shared_link_with_settings(f'/{filename}')
        
        # Return the URL
        return shared_link.url
        
    except dropbox.exceptions.AuthError as e:
        return f"Dropbox authentication error: {str(e)}"
    except dropbox.exceptions.ApiError as e:
        return f"Dropbox API error: {str(e)}"
    except Exception as e:
        return f"Unexpected error: {str(e)}"

def convert_dropbox_url_to_direct(dropbox_url):
    """Convert Dropbox shareable URL to direct image URL for display"""
    if not dropbox_url or "Error:" in dropbox_url:
        return dropbox_url
    
    # Convert from: https://www.dropbox.com/s/abc123/filename.png?dl=0
    # To: https://dl.dropboxusercontent.com/s/abc123/filename.png
    if "dropbox.com/s/" in dropbox_url:
        direct_url = dropbox_url.replace("www.dropbox.com/s/", "dl.dropboxusercontent.com/s/")
        direct_url = direct_url.split("?")[0]  # Remove query parameters
        return direct_url
    
    return dropbox_url

def create_life_expectancy_plot():
    """Create life expectancy plot and return image data"""
    # Create dummy data for life expectancy in Denmark (2013-2020)
    years = list(range(2013, 2021))
    life_expectancy = [80.2, 80.4, 80.6, 80.8, 81.0, 81.2, 81.4, 81.6]
    
    # Create the plot
    plt.figure(figsize=(6, 4))
    plt.plot(years, life_expectancy, marker='o', linewidth=2, markersize=5, color='#2E86AB')
    plt.title('Life Expectancy in Denmark (2013-2020)', fontsize=12, fontweight='bold', pad=10)
    plt.xlabel('Year', fontsize=9)
    plt.ylabel('Life Expectancy (years)', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.xticks(years)
    
    # Add value labels on data points
    for year, value in zip(years, life_expectancy):
        plt.annotate(f'{value:.1f}', (year, value), textcoords="offset points", 
                    xytext=(0,6), ha='center', fontsize=7)
    
    plt.tight_layout()
    
    # Save to BytesIO buffer
    buffer = BytesIO()
    plt.savefig(buffer, format='png', dpi=100, bbox_inches='tight')
    buffer.seek(0)
    image_data = buffer.getvalue()
    plt.close()
    
    return image_data

def plot_indicator(indicator_code: str, country_code: str) -> str:
    """Plot indicator and upload to Dropbox, return URL for chat display"""
    try:
        # Create the plot
        image_data = create_life_expectancy_plot()
        
        # Generate filename with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"life_expectancy_{country_code}_{timestamp}.png"
        
        # Upload to Dropbox and get URL
        dropbox_url = upload_image_to_dropbox(image_data, filename)
        
        # Convert to direct URL for better display
        direct_url = convert_dropbox_url_to_direct(dropbox_url)
        
        # Return HTML img tag for chat display
        return f'<img src="{direct_url}" alt="Life Expectancy Chart" style="max-width: 100%; height: auto;">'
        
    except Exception as e:
        return f"Error creating plot: {str(e)}"


In [ ]:
# Test Class for Plot and Dropbox Functions
class TestPlotAndDropboxFunctions(unittest.TestCase):
    
    def setUp(self):
        """Set up test fixtures before each test method."""
        self.test_image_data = b"fake_image_data_for_testing"
        self.test_filename = "test_image.png"
        self.test_country_code = "DK"
        self.test_indicator_code = "life_expectancy"
    
    def test_create_life_expectancy_plot_returns_bytes(self):
        """Test that create_life_expectancy_plot returns image data as bytes."""
        image_data = create_life_expectancy_plot()
        
        # Check that we get bytes data
        self.assertIsInstance(image_data, bytes)
        self.assertGreater(len(image_data), 0)
        
        # Check that it's valid PNG data (starts with PNG signature)
        self.assertTrue(image_data.startswith(b'\x89PNG'))
    
    def test_convert_dropbox_url_to_direct_valid_url(self):
        """Test URL conversion with valid Dropbox URL."""
        test_url = "https://www.dropbox.com/s/abc123/test.png?dl=0"
        expected_url = "https://dl.dropboxusercontent.com/s/abc123/test.png"
        
        result = convert_dropbox_url_to_direct(test_url)
        self.assertEqual(result, expected_url)
    
    def test_convert_dropbox_url_to_direct_error_url(self):
        """Test URL conversion with error URL."""
        error_url = "Error: DROPBOX_ACCESS_TOKEN not found"
        result = convert_dropbox_url_to_direct(error_url)
        self.assertEqual(result, error_url)
    
    def test_convert_dropbox_url_to_direct_none_url(self):
        """Test URL conversion with None URL."""
        result = convert_dropbox_url_to_direct(None)
        self.assertIsNone(result)
    
    def test_convert_dropbox_url_to_direct_invalid_url(self):
        """Test URL conversion with invalid URL format."""
        invalid_url = "https://example.com/test.png"
        result = convert_dropbox_url_to_direct(invalid_url)
        self.assertEqual(result, invalid_url)


In [ ]:
    @patch('os.getenv')
    def test_upload_image_to_dropbox_no_token(self, mock_getenv):
        """Test upload_image_to_dropbox when no access token is found."""
        mock_getenv.return_value = None
        
        result = upload_image_to_dropbox(self.test_image_data, self.test_filename)
        
        self.assertEqual(result, "Error: DROPBOX_ACCESS_TOKEN not found in environment variables")
        mock_getenv.assert_called_once_with('DROPBOX_ACCESS_TOKEN')
    
    @patch('dropbox.Dropbox')
    @patch('os.getenv')
    def test_upload_image_to_dropbox_success(self, mock_getenv, mock_dropbox_class):
        """Test successful upload to Dropbox."""
        # Mock environment variable
        mock_getenv.return_value = "fake_access_token"
        
        # Mock Dropbox client and its methods
        mock_dropbox_instance = Mock()
        mock_dropbox_class.return_value = mock_dropbox_instance
        
        # Mock the shared link response
        mock_shared_link = Mock()
        mock_shared_link.url = "https://www.dropbox.com/s/abc123/test.png?dl=0"
        mock_dropbox_instance.sharing_create_shared_link_with_settings.return_value = mock_shared_link
        
        result = upload_image_to_dropbox(self.test_image_data, self.test_filename)
        
        # Verify the result
        self.assertEqual(result, "https://www.dropbox.com/s/abc123/test.png?dl=0")
        
        # Verify method calls
        mock_dropbox_instance.files_upload.assert_called_once_with(
            self.test_image_data, f'/{self.test_filename}'
        )
        mock_dropbox_instance.sharing_create_shared_link_with_settings.assert_called_once_with(
            f'/{self.test_filename}'
        )
    
    @patch('dropbox.Dropbox')
    @patch('os.getenv')
    def test_upload_image_to_dropbox_auth_error(self, mock_getenv, mock_dropbox_class):
        """Test upload_image_to_dropbox with authentication error."""
        mock_getenv.return_value = "fake_access_token"
        
        # Mock Dropbox to raise AuthError
        mock_dropbox_instance = Mock()
        mock_dropbox_class.return_value = mock_dropbox_instance
        mock_dropbox_instance.files_upload.side_effect = dropbox.exceptions.AuthError(
            "request_id", "error", "user_message"
        )
        
        result = upload_image_to_dropbox(self.test_image_data, self.test_filename)
        
        self.assertTrue(result.startswith("Dropbox authentication error:"))
    
    @patch('dropbox.Dropbox')
    @patch('os.getenv')
    def test_upload_image_to_dropbox_api_error(self, mock_getenv, mock_dropbox_class):
        """Test upload_image_to_dropbox with API error."""
        mock_getenv.return_value = "fake_access_token"
        
        # Mock Dropbox to raise ApiError
        mock_dropbox_instance = Mock()
        mock_dropbox_class.return_value = mock_dropbox_instance
        mock_dropbox_instance.files_upload.side_effect = dropbox.exceptions.ApiError(
            "request_id", "error", "user_message"
        )
        
        result = upload_image_to_dropbox(self.test_image_data, self.test_filename)
        
        self.assertTrue(result.startswith("Dropbox API error:"))


In [ ]:
    @patch('__main__.upload_image_to_dropbox')
    @patch('__main__.create_life_expectancy_plot')
    @patch('__main__.convert_dropbox_url_to_direct')
    def test_plot_indicator_success(self, mock_convert_url, mock_create_plot, mock_upload):
        """Test successful plot_indicator function."""
        mock_image_data = b"fake_plot_data"
        mock_create_plot.return_value = mock_image_data
        mock_dropbox_url = "https://www.dropbox.com/s/abc123/test.png?dl=0"
        mock_upload.return_value = mock_dropbox_url
        mock_direct_url = "https://dl.dropboxusercontent.com/s/abc123/test.png"
        mock_convert_url.return_value = mock_direct_url
        
        result = plot_indicator(self.test_indicator_code, self.test_country_code)
        
        expected_html = f'<img src="{mock_direct_url}" alt="Life Expectancy Chart" style="max-width: 100%; height: auto;">'
        self.assertEqual(result, expected_html)
        mock_create_plot.assert_called_once()
        mock_upload.assert_called_once()
        mock_convert_url.assert_called_once_with(mock_dropbox_url)
    
    @patch('__main__.create_life_expectancy_plot')
    def test_plot_indicator_plot_creation_error(self, mock_create_plot):
        """Test plot_indicator when plot creation fails."""
        mock_create_plot.side_effect = Exception("Plot creation failed")
        result = plot_indicator(self.test_indicator_code, self.test_country_code)
        self.assertTrue(result.startswith("Error creating plot:"))
        self.assertIn("Plot creation failed", result)
    
    @patch('__main__.upload_image_to_dropbox')
    @patch('__main__.create_life_expectancy_plot')
    def test_plot_indicator_upload_error(self, mock_create_plot, mock_upload):
        """Test plot_indicator when upload fails."""
        mock_create_plot.return_value = b"fake_plot_data"
        mock_upload.return_value = "Error: Upload failed"
        result = plot_indicator(self.test_indicator_code, self.test_country_code)
        self.assertIn('<img src="Error: Upload failed"', result)
        self.assertIn('alt="Life Expectancy Chart"', result)
    
    def test_plot_indicator_filename_generation(self):
        """Test that plot_indicator generates correct filename format."""
        with patch('__main__.upload_image_to_dropbox') as mock_upload, \
             patch('__main__.create_life_expectancy_plot') as mock_create_plot, \
             patch('__main__.convert_dropbox_url_to_direct') as mock_convert_url:
            
            mock_create_plot.return_value = b"fake_data"
            mock_upload.return_value = "https://dropbox.com/test.png"
            mock_convert_url.return_value = "https://dropboxusercontent.com/test.png"
            
            plot_indicator("life_expectancy", "DK")
            
            upload_call_args = mock_upload.call_args[0]
            filename = upload_call_args[1]
            
            self.assertTrue(filename.startswith("life_expectancy_DK_"))
            self.assertTrue(filename.endswith(".png"))
            
            timestamp_part = filename.split("_")[-1].replace(".png", "")
            self.assertEqual(len(timestamp_part), 14)


In [ ]:
# Test Runner
def run_tests():
    """Run all unit tests and display results."""
    # Create test suite
    test_suite = unittest.TestLoader().loadTestsFromTestCase(TestPlotAndDropboxFunctions)
    
    # Run tests with detailed output
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(test_suite)
    
    # Print summary
    print(f"\n{'='*50}")
    print(f"Test Summary:")
    print(f"Tests run: {result.testsRun}")
    print(f"Failures: {len(result.failures)}")
    print(f"Errors: {len(result.errors)}")
    print(f"Success rate: {((result.testsRun - len(result.failures) - len(result.errors)) / result.testsRun * 100):.1f}%")
    
    if result.failures:
        print(f"\nFailures:")
        for test, traceback in result.failures:
            print(f"- {test}: {traceback}")
    
    if result.errors:
        print(f"\nErrors:")
        for test, traceback in result.errors:
            print(f"- {test}: {traceback}")
    
    return result

# Run the tests
if __name__ == "__main__":
    test_result = run_tests()


In [15]:
# Real Dropbox API Test - Upload and Display URL
def test_real_dropbox_upload():
    """Test real Dropbox API upload with actual image and display URL."""
    print("🚀 Testing Real Dropbox API Upload...")
    print("=" * 50)
    
    # Check for access token
    access_token = os.getenv('DROPBOX_ACCESS_TOKEN')
    if not access_token or access_token == "your_access_token_here":
        print("❌ Error: DROPBOX_ACCESS_TOKEN not found in environment variables")
        print("   Please set DROPBOX_ACCESS_TOKEN in your .env file")
        return False
    
    try:
        # Step 1: Create test image
        print("1️⃣ Creating test image...")
        image_data = create_life_expectancy_plot()
        print(f"   ✅ Image created: {len(image_data)} bytes")
        
        # Step 2: Generate unique filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"test_upload_{timestamp}.png"
        print(f"   📁 Filename: {filename}")
        
        # Step 3: Upload to Dropbox
        print("2️⃣ Uploading to Dropbox...")
        dropbox_url = upload_image_to_dropbox(image_data, filename)
        
        if dropbox_url.startswith("Error:"):
            print(f"   ❌ Upload failed: {dropbox_url}")
            return False
        
        print(f"   ✅ Upload successful!")
        print(f"   🔗 Dropbox URL: {dropbox_url}")
        
        # Step 4: Convert to direct URL
        print("3️⃣ Converting to direct URL...")
        direct_url = convert_dropbox_url_to_direct(dropbox_url)
        print(f"   🔗 Direct URL: {direct_url}")
        
        # Step 5: Test full workflow
        print("4️⃣ Testing complete workflow...")
        result = plot_indicator("life_expectancy", "DK")
        
        if '<img src=' in result and 'alt="Life Expectancy Chart"' in result:
            print("   ✅ Complete workflow successful!")
            print(f"   📄 HTML Result: {result[:100]}...")
        else:
            print("   ❌ Complete workflow failed")
            return False
        
        print("\n🎉 All tests passed!")
        print("=" * 50)
        print("📋 Summary:")
        print(f"   • Image size: {len(image_data)} bytes")
        print(f"   • Filename: {filename}")
        print(f"   • Dropbox URL: {dropbox_url}")
        print(f"   • Direct URL: {direct_url}")
        
        return True
        
    except Exception as e:
        print(f"❌ Test failed with error: {str(e)}")
        return False

# Run the real test
if __name__ == "__main__":
    test_real_dropbox_upload()


🚀 Testing Real Dropbox API Upload...
1️⃣ Creating test image...
   ✅ Image created: 34322 bytes
   📁 Filename: test_upload_20250926_113845.png
2️⃣ Uploading to Dropbox...
   ✅ Upload successful!
   🔗 Dropbox URL: https://www.dropbox.com/scl/fi/mmdwq1j0s0icv5ozpp4ne/test_upload_20250926_113845.png?rlkey=ifagvffw2m6e5mkikf9cu78g1&dl=0
3️⃣ Converting to direct URL...
   🔗 Direct URL: https://www.dropbox.com/scl/fi/mmdwq1j0s0icv5ozpp4ne/test_upload_20250926_113845.png?rlkey=ifagvffw2m6e5mkikf9cu78g1&dl=0
4️⃣ Testing complete workflow...
   ✅ Complete workflow successful!
   📄 HTML Result: <img src="https://www.dropbox.com/scl/fi/56sk1nhfl31khsczofdtr/life_expectancy_DK_20250926_113849.pn...

🎉 All tests passed!
📋 Summary:
   • Image size: 34322 bytes
   • Filename: test_upload_20250926_113845.png
   • Dropbox URL: https://www.dropbox.com/scl/fi/mmdwq1j0s0icv5ozpp4ne/test_upload_20250926_113845.png?rlkey=ifagvffw2m6e5mkikf9cu78g1&dl=0
   • Direct URL: https://www.dropbox.com/scl/fi/mmdwq1j0

In [ ]:
# Execute Real Dropbox Test
# Uncomment the line below to run the real Dropbox API test
# test_real_dropbox_upload()
